<a href="https://colab.research.google.com/github/jayanthkorupolu2000-web/Final_Project/blob/main/Vit_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
from tensorflow.keras import layers, models

In [4]:
import os

BASE_PATH = "/content/drive/MyDrive/crack_dataset"

print("Train exists:", os.path.exists(BASE_PATH + "/train"))
print("Valid exists:", os.path.exists(BASE_PATH + "/valid"))
print("Test exists :", os.path.exists(BASE_PATH + "/test"))

print("Train annotation file:",
      os.path.exists(BASE_PATH + "/train/_annotations.coco.json"))


Train exists: True
Valid exists: True
Test exists : True
Train annotation file: True


In [10]:
#Read COCO annotations (TRAIN only) & verify contents
import json

ANNOTATION_FILE = BASE_PATH + "/train/_annotations.coco.json"

with open(ANNOTATION_FILE, "r") as f:
    coco = json.load(f)

# Basic counts
num_images = len(coco["images"])
num_annotations = len(coco["annotations"])
num_categories = len(coco["categories"])

print("Number of images      :", num_images)
print("Number of annotations :", num_annotations)
print("Number of categories  :", num_categories)

# Show first 5 image entries
print("\nFirst 5 images:")
for img in coco["images"][:5]:
    print("  ", img["id"], "→", img["file_name"])

# Show first 5 bounding boxes
print("\nFirst 5 annotations (bbox format):")
for ann in coco["annotations"][:5]:
    print("  Image ID:", ann["image_id"], "| BBox:", ann["bbox"])


Number of images      : 735
Number of annotations : 4233
Number of categories  : 2

First 5 images:
   0 → article-0-1EB878FE00000578-852_634x423_jpg.rf.f4f0555a56808adb096aaded6812d891.jpg
   1 → im191_jpg.rf.f84ec39898d0ce6b4d3770b11f7f74d7.jpg
   2 → cd_2608_13_jpg.rf.f74d954858ba9bdfc8af71c26cd3da09.jpg
   3 → im255_jpg.rf.f4dfe53ec9e88f0ba6994fa6efa78c25.jpg
   4 → bottom-repair_jpg.rf.f53dbf04298cb781460ea24874616a2d.jpg

First 5 annotations (bbox format):
  Image ID: 0 | BBox: [24, 110, 48, 24]
  Image ID: 0 | BBox: [56, 35, 60, 48]
  Image ID: 0 | BBox: [138, 0, 128, 72]
  Image ID: 0 | BBox: [277, 453, 24, 36]
  Image ID: 0 | BBox: [115, 528, 24, 30]


In [11]:
# Map image_id -> file_name
img_id_to_name = {img["id"]: img["file_name"] for img in coco["images"]}

# Initialize all images as no-crack
labels = {img["file_name"]: 0 for img in coco["images"]}

# Mark images having at least one bbox as crack
for ann in coco["annotations"]:
    img_name = img_id_to_name[ann["image_id"]]
    labels[img_name] = 1

# Count labels
crack_count = sum(labels.values())
no_crack_count = len(labels) - crack_count

print("Crack images    :", crack_count)
print("No-crack images :", no_crack_count)


Crack images    : 735
No-crack images : 0


In [12]:
# Collect bbox areas
areas = []
for ann in coco["annotations"]:
    _, _, w, h = ann["bbox"]
    areas.append(w * h)

print("Min area :", min(areas))
print("Max area :", max(areas))
print("Avg area :", sum(areas) / len(areas))


Min area : 8
Max area : 290560
Avg area : 7655.307583274273


In [13]:
from PIL import Image

TRAIN_IMG_DIR = BASE_PATH + "/train"
MIN_CRACK_AREA = 50

positive_crops = []

for ann in coco["annotations"]:
    x, y, w, h = map(int, ann["bbox"])
    if w * h < MIN_CRACK_AREA:
        continue

    img_name = img_id_to_name[ann["image_id"]]
    img_path = os.path.join(TRAIN_IMG_DIR, img_name)

    if not os.path.exists(img_path):
        continue

    img = Image.open(img_path).convert("RGB")
    crop = img.crop((x, y, x + w, y + h))
    positive_crops.append(crop)

print("Positive crack samples:", len(positive_crops))


Positive crack samples: 3824


In [14]:
import random

NEGATIVE_SAMPLES = 3824  # match positives
CROP_SIZE = 64           # background crop size

negative_crops = []

# Group annotations by image
from collections import defaultdict
img_to_bboxes = defaultdict(list)

for ann in coco["annotations"]:
    img_to_bboxes[ann["image_id"]].append(ann["bbox"])

# Generate background crops
for img_id, bboxes in img_to_bboxes.items():
    img_name = img_id_to_name[img_id]
    img_path = os.path.join(TRAIN_IMG_DIR, img_name)

    if not os.path.exists(img_path):
        continue

    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    attempts = 0
    while attempts < 10 and len(negative_crops) < NEGATIVE_SAMPLES:
        x = random.randint(0, max(0, W - CROP_SIZE))
        y = random.randint(0, max(0, H - CROP_SIZE))

        # check overlap with any crack bbox
        overlap = False
        for bx, by, bw, bh in bboxes:
            if not (x + CROP_SIZE < bx or x > bx + bw or
                    y + CROP_SIZE < by or y > by + bh):
                overlap = True
                break

        if not overlap:
            crop = img.crop((x, y, x + CROP_SIZE, y + CROP_SIZE))
            negative_crops.append(crop)

        attempts += 1

    if len(negative_crops) >= NEGATIVE_SAMPLES:
        break

print("Negative no-crack samples:", len(negative_crops))


Negative no-crack samples: 3824


In [15]:
# Combine data
X = positive_crops + negative_crops
y = [1] * len(positive_crops) + [0] * len(negative_crops)

print("Total samples:", len(X))
print("Crack samples :", sum(y))
print("No-crack samples:", len(y) - sum(y))


Total samples: 7648
Crack samples : 3824
No-crack samples: 3824


In [16]:
import tensorflow as tf
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 16

# TF preprocessing (resize + normalize)
def preprocess_tf(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# Generator to avoid loading all data into RAM
def data_generator():
    for img, label in zip(X, y):
        yield np.array(img), label

# Create TensorFlow dataset safely
dataset = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
)

dataset = (
    dataset
    .map(preprocess_tf)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("TensorFlow dataset ready (runtime safe)")


TensorFlow dataset ready (runtime safe)


In [17]:
# Train / validation split (generator-safe)
TRAIN_BATCHES = int(0.8 * 7648 / 16)

train_ds = dataset.take(TRAIN_BATCHES)
val_ds = dataset.skip(TRAIN_BATCHES)

print("Train/Val split done")

Train/Val split done


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Basic configuration
IMG_SIZE = 224
PATCH_SIZE = 16                      # Image split into 16x16 patches
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
EMBED_DIM = 64                       # Patch embedding size
NUM_HEADS = 4                        # Attention heads
FF_DIM = 128                         # Feed-forward layer size
NUM_LAYERS = 4                       # Transformer blocks
NUM_CLASSES = 1                      # Binary classification


# Converts image into patch embeddings
class PatchEmbedding(layers.Layer):
    def __init__(self):
        super().__init__()
        self.proj = layers.Conv2D(
            EMBED_DIM, PATCH_SIZE, PATCH_SIZE
        )                            # Patch projection
        self.flatten = layers.Reshape(
            (NUM_PATCHES, EMBED_DIM)
        )

    def call(self, x):
        x = self.proj(x)             # Create patch embeddings
        return self.flatten(x)       # Shape: (num_patches, embed_dim)


# Single transformer encoder block
def transformer_block(x):
    # Self-attention on patches
    attn = layers.MultiHeadAttention(
        NUM_HEADS, EMBED_DIM
    )(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    # Feed-forward network
    ffn = layers.Dense(FF_DIM, activation="relu")(x)
    ffn = layers.Dense(EMBED_DIM)(ffn)
    x = layers.Add()([x, ffn])
    return layers.LayerNormalization()(x)


# Vision Transformer model
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = PatchEmbedding()(inputs)         # Image → patches

# Positional encoding (keeps spatial order)
positions = tf.range(start=0, limit=NUM_PATCHES, delta=1)
pos_embed = layers.Embedding(
    NUM_PATCHES, EMBED_DIM
)(positions)
x = x + pos_embed

# Stack transformer encoder blocks
for _ in range(NUM_LAYERS):
    x = transformer_block(x)

# Global representation of image
x = layers.GlobalAveragePooling1D()(x)

# Binary classification output
outputs = layers.Dense(
    NUM_CLASSES, activation="sigmoid"
)(x)

vit_model = models.Model(inputs, outputs)


In [ ]:
# Compile model
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # stable for ViT
    loss="binary_crossentropy",                               # binary labels
    metrics=["accuracy"]
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/vit_checkpoint.keras",
    save_best_only=True,
    monitor="val_loss"
)


# Train model
history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_cb]
)

# 👇 PUT THIS IMMEDIATELY AFTER TRAINING
vit_model.save("/content/drive/MyDrive/vit_model.keras")
print("Model saved to Drive")





Epoch 1/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 35s 46ms/step - accuracy: 0.8186 - loss: 0.4040 - val_accuracy: 0.9733 - val_loss: 0.0988
Epoch 2/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 34s 43ms/step - accuracy: 0.8233 - loss: 0.3891 - val_accuracy: 0.9701 - val_loss: 0.1284
Epoch 3/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.8266 - loss: 0.3510 - val_accuracy: 0.9681 - val_loss: 0.1100
Epoch 4/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 16s 41ms/step - accuracy: 0.8392 - loss: 0.3459 - val_accuracy: 0.9766 - val_loss: 0.0955
Epoch 5/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - accuracy: 0.8199 - loss: 0.3924 - val_accuracy: 0.9661 - val_loss: 0.1096
Epoch 6/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.8319 - loss: 0.3845 - val_accuracy: 0.9701 - val_loss: 0.1259
Epoch 7/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.8181 - loss: 0.3604 - val_accuracy: 0.9629 - val_loss: 0.1157
Epoch 8/10
382/382 ━━━━━━━━━━━━━━━━━━━━ 16s 40ms/step - accuracy: 0.8369 - loss: 0.3635 - 